# SoccerTrack v2 — quickstart

Load one match's **GSR** (game state reconstruction) and **BAS** (ball action spotting) annotations, plot a pitch frame with players, and list events.

Prereqs:
- Clone the repo: `git clone https://github.com/AtomScott/SoccerTrack-v2.git && cd SoccerTrack-v2`
- Install: `uv sync` (Python 3.12+).
- Download at least one match from Hugging Face or Google Drive into `./data/` with the standard layout (see [`docs/format-gsr.md`](../docs/format-gsr.md)).

If you don't yet have data on disk, set `DATA_ROOT` to point at your local snapshot — the loader reads plain JSON files.


In [ ]:
from pathlib import Path
import sys

# Make src/ importable when running from notebooks/
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_utils.soccertrack_v2 import load_match, list_matches, BAS_LABELS

DATA_ROOT = REPO_ROOT / "data"  # adjust if your download lives elsewhere
print("matches found:", list_matches(DATA_ROOT))

In [ ]:
MATCH_ID = "117093"  # or the first id from list_matches(DATA_ROOT)
match = load_match(DATA_ROOT, MATCH_ID)
frames = match.gsr_frames(half=1)
print(f"{len(frames)} annotated frames in half 1")
print("first frame:", frames[0].image_id, "last:", frames[-1].image_id)
print("entities at frame 0:", len(frames[0].entities))

## Plot a single frame on a pitch

`mplsoccer` is already a dependency (see `pyproject.toml`).


In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch

frame = frames[len(frames) // 2]

# SoccerTrack v2 pitch coords are centred at (0, 0) in metres. mplsoccer defaults to (0, 0) bottom-left,
# so we shift by half-pitch dimensions when plotting.
PITCH_W, PITCH_H = 105.0, 68.0
pitch = Pitch(pitch_type="custom", pitch_length=PITCH_W, pitch_width=PITCH_H, line_color="#555")
fig, ax = pitch.draw(figsize=(10, 6))

role_colour = {"player": "#10b981", "goalkeeper": "#ff6b35", "referee": "#444", "other": "#999"}
for p in frame.entities:
    xp, yp = p.x + PITCH_W / 2, p.y + PITCH_H / 2
    ax.scatter(xp, yp, s=180, c=role_colour.get(p.role, "#000"), edgecolors="white", linewidths=1.2, zorder=3)
    if p.jersey_number is not None:
        ax.text(xp, yp, str(p.jersey_number), color="white", ha="center", va="center", fontsize=8, zorder=4)

ax.set_title(f"match {MATCH_ID} half 1  frame {frame.image_id}  t={frame.t_ms/1000:.1f}s")
plt.show()

## BAS: list events and count per class


In [ ]:
from collections import Counter

events = match.bas_events()
print(f"{len(events)} events total")
counts = Counter(e.label for e in events)
for label in BAS_LABELS:
    print(f"  {label:<28} {counts.get(label, 0)}")

# Preview the first few shots
for ev in match.events_of("Shot")[:5]:
    print(ev.half, ev.clock, ev.t_ms, ev.team, ev.player_id)

## Cross-reference: GSR frame at a BAS event

Given an event, the corresponding GSR frame is `round(t_ms / 40)` on the matching half file.


In [ ]:
shots = match.events_of("Shot")
if shots:
    ev = shots[0]
    f = match.gsr_frame(half=ev.half, image_id=ev.image_id)
    print("event", ev.label, ev.clock, "team", ev.team)
    if f is not None:
        print(f"  {len(f.entities)} entities observed at this frame")
    else:
        print("  (no GSR record at this exact frame — check ±1 frame)")

## Next steps

- Evaluation CLIs: `python -m src.evaluation.gs_hota --pred ... --gt ...` (GSR), `python -m src.evaluation.bas_map --pred ... --gt ...` (BAS).
- Baseline starter kits: see [`baselines/gsr/`](../baselines/gsr/), [`baselines/bas/`](../baselines/bas/), [`baselines/mot/`](../baselines/mot/).
- Leaderboard + submission flow: [`docs/leaderboard.html`](../docs/leaderboard.html).

File a bug or dataset issue on GitHub if anything in this notebook didn't work.